In [5]:
!pip install kaggle

import os
os.makedirs('/root/.kaggle', exist_ok=True)

# upload kaggle.json first
from google.colab import files
files.upload()

!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# download dataset
!kaggle datasets download -d kazanova/sentiment140

# unzip
!unzip sentiment140.zip

Saving kaggle .json to kaggle  (1).json
cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/kazanova/sentiment140
License(s): other
100% 80.9M/80.9M [00:00<00:00, 270MB/s]

Archive:  sentiment140.zip
  inflating: training.1600000.processed.noemoticon.csv  


In [6]:
import pandas as pd

df = pd.read_csv('training.1600000.processed.noemoticon.csv', encoding='ISO-8859-1')

# take small sample
df = df.sample(20000)

df.to_csv('sample_data.csv', index=False)

In [7]:
file_path = "sample_data.csv"

In [8]:
# ==============================
# Tweet Sentiment Analysis
# ==============================

import pandas as pd
import numpy as np
import re
import pickle
import nltk

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# ------------------------------
# 1. Download Resources
# ------------------------------
nltk.download('stopwords')

# ------------------------------
# 2. Load Dataset
# ------------------------------
def load_data(path):
    column_names = ['target', 'id', 'date', 'flag', 'user', 'text']
    data = pd.read_csv(path, names=column_names, encoding='ISO-8859-1')

    # Convert 4 → 1 (positive)
    data.replace({'target': {4: 1}}, inplace=True)

    return data

# ------------------------------
# 3. Text Preprocessing
# ------------------------------
port_stem = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = text.lower()
    text = text.split()

    text = [port_stem.stem(word) for word in text if word not in stop_words]

    return ' '.join(text)

# ------------------------------
# 4. Prepare Data
# ------------------------------
def prepare_data(data):
    print("Preprocessing text... (this may take time)")

    data['processed_text'] = data['text'].apply(preprocess_text)

    X = data['processed_text'].values
    Y = data['target'].values

    return X, Y

# ------------------------------
# 5. Train Model
# ------------------------------
def train_model(X, Y):
    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y, test_size=0.2, stratify=Y, random_state=2
    )

    vectorizer = TfidfVectorizer()

    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, Y_train)

    # Accuracy
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    print("\nModel Performance:")
    print("Training Accuracy:", accuracy_score(Y_train, train_pred))
    print("Test Accuracy:", accuracy_score(Y_test, test_pred))

    return model, vectorizer

# ------------------------------
# 6. Save Model
# ------------------------------
def save_model(model, vectorizer):
    pickle.dump(model, open('model.pkl', 'wb'))
    pickle.dump(vectorizer, open('vectorizer.pkl', 'wb'))
    print("\nModel and vectorizer saved successfully!")

# ------------------------------
# 7. Prediction Function
# ------------------------------
def predict_sentiment(text, model, vectorizer):
    text = preprocess_text(text)
    text = vectorizer.transform([text])

    prediction = model.predict(text)[0]

    return "Positive 😊" if prediction == 1 else "Negative 😞"

# ------------------------------
# 8. Main Function
# ------------------------------
def main():
    # Change path to your dataset file
    file_path = "training.1600000.processed.noemoticon.csv"

    data = load_data(file_path)

    # Optional: reduce size for faster execution
    data = data.sample(50000, random_state=2)

    X, Y = prepare_data(data)

    model, vectorizer = train_model(X, Y)

    save_model(model, vectorizer)

    # Test prediction
    print("\nSample Predictions:")
    print(predict_sentiment("I love this product!", model, vectorizer))
    print(predict_sentiment("This is the worst experience ever.", model, vectorizer))

# ------------------------------
# Run Program
# ------------------------------
if __name__ == "__main__":
    main()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Preprocessing text... (this may take time)

Model Performance:
Training Accuracy: 0.835125
Test Accuracy: 0.7539

Model and vectorizer saved successfully!

Sample Predictions:
Positive 😊
Negative 😞


In [16]:
import pickle

model = pickle.load(open('model.pkl', 'rb'))
vectorizer = pickle.load(open('vectorizer.pkl', 'rb'))

In [17]:
def predict(text):
    text = preprocess_text(text)   # correct name
    text = vectorizer.transform([text])
    result = model.predict(text)[0]
    return "Positive 😊" if result == 1 else "Negative 😞"

In [ ]:
while True:
    user_input = input("\nEnter a tweet (or type 'exit'): ")

    if user_input.lower() == 'exit':
        break

    print("Prediction:", predict(user_input))


Enter a tweet (or type 'exit'): i like this movie 
Prediction: Positive 😊
